# Tune data cleaning with a matrix

Run one [`quality`](../reference/presets.md#quality) entry over a grid of outlier thresholds and near-duplicate
sensitivities, and compare what each combination finds in one table.

You are a T&E engineer or data scientist who tunes a quality configuration and needs to see how sensitive its findings
are to the detection thresholds before choosing operational settings.

You can run a matrix during the data-quality stage of your T&E workflow. Instead of guessing thresholds for
{doc}`data_cleaning`, run a range of them, compare the findings side by side, and choose settings you can defend. You
then apply those settings in an operational cleaning run. See {doc}`../concepts/Reproducibility` for how the runs
share one draw of the data and one cache.

## What you will do

- Load a subset of the MilitaryVehicles classification dataset.
- Configure a `quality` entry, and a task whose `matrix` varies two of its settings.
- Run the task once per combination, sharing the statistics, embeddings and clusters across runs.
- Read the comparison table, and one run's result in full.
- Choose quality settings from what the table shows.

## What you will learn

- How a task's `matrix` runs its entry once per combination of the values it lists.
- How to read a matrix result's comparison table, and reach each run's result.
- How to tell a setting that changes the findings from one that changes nothing on your data.

## Prerequisites

- Install `dataeval-flow`, which includes `dataeval`, `datamaite` and `pydantic`.
- Install `maite-datasets` to download MilitaryVehicles.
- Make sure you have network access for the initial dataset download.

## Lay out the data root

The data root is the folder you run this notebook from. After a first run it holds:

```text
project/
├── data/
│   ├── militaryvehicles/                    # downloaded on the first run (~141 MB); the export is made from it
│   └── militaryvehicles_datamaite_train/    # exported on the first run; read: the config points at its train/
└── cache/                                   # written: statistics, embeddings and clusters, shared by all nine runs
```

The notebook builds its pipeline in Python and reads no configuration file. The config's dataset path,
`data/militaryvehicles_datamaite_train/train`, is relative to the data root. It ends in `train/` because the export
nests its images under the split name.

## Load the dataset

[MilitaryVehicles](https://huggingface.co/datasets/leibnitz-lab/military_vehicles) is a classification dataset
containing 9,444 images across 24 vehicle types. The dataset includes varying image resolutions, diverse visual
backgrounds, and exact duplicate images.

Use `maite-datasets` to download the dataset. Setting `as_datamaite=True` writes the data in class-per-directory
ImageFolder format, which you can load with `task="image_classification"`.

:::{note}
The download is approximately 141 MB across 9,466 files. Set `HF_TOKEN` in your environment to enable parallel
downloads and avoid anonymous rate limits.
:::

In [ ]:
from pathlib import Path

from maite_datasets.image_classification import MilitaryVehicles

data_root = Path("./data")

# Download once; subsequent runs read existing disk files.
MilitaryVehicles(root=data_root, image_set="base", download=True)
MilitaryVehicles(root=data_root, image_set="train", as_datamaite=True)

# The export nests images under the split directory.
data_path = data_root / "militaryvehicles_datamaite_train" / "train"
print(f"Reading from {data_path}")

## Build the configuration

The `quality` entry is an ordinary one, and must be valid on its own. It sets `outliers.outlier_threshold` and
`outliers.flags`, which have no defaults. Its cluster settings, `duplicates.cluster_algorithm` here, turn on
cluster-based near-duplicate detection, which reads the BoVW embeddings.

The task's `matrix` names the settings to vary and the values each takes:

- `outliers.outlier_threshold` takes `adaptive` with the bound 2.5 (aggressive), 3.5 and 4.5 (conservative).
- `duplicates.cluster_sensitivity` takes 0.5, 2.0 and 3.0, within the 0.1 to 3.0 that DataEval documents as its
  typical range.

One grid crosses every key with every other, so the task runs 3 × 3 = 9 times. The entry's own values for these two
settings, if it set any, would be replaced in each run.

In [ ]:
from dataeval_flow import run_task
from dataeval_flow.config import (
    HuggingFaceDatasetConfig,
    PipelineConfig,
    SourceConfig,
    TaskConfig,
    ViewConfig,
    ViewOperation,
)
from dataeval_flow.config.extractors import BoVWExtractorConfig
from dataeval_flow.workflows.quality import QualityConfig

cleaning = QualityConfig(
    name="mv_cleaning",
    outliers={"flags": ["dimension", "pixel", "visual"], "outlier_threshold": "adaptive"},
    duplicates={"cluster_algorithm": "hdbscan", "merge_near_duplicates": True},
)

task = TaskConfig(
    name="mv_tuning",
    workflow="mv_cleaning",
    sources="mv_src",
    extractor="bovw_ext",
    matrix={
        "outliers.outlier_threshold": [["adaptive", 2.5], ["adaptive", 3.5], ["adaptive", 4.5]],
        "duplicates.cluster_sensitivity": [0.5, 2.0, 3.0],
    },
)

config = PipelineConfig(
    max_processes=2,
    # Each run starts from this seed, so no run's clustering depends on the runs before it.
    seed=42,
    datasets=[
        HuggingFaceDatasetConfig(name="mv_train", path=str(data_path), task="image_classification"),
    ],
    views=[
        # Shuffle before limiting to avoid sampling only the first alphabetical classes.
        ViewConfig(
            name="sample1000",
            operations=[
                ViewOperation(type="Shuffle", params={"seed": 42}),
                ViewOperation(type="Limit", params={"size": 1000}),
            ],
        ),
    ],
    sources=[
        SourceConfig(name="mv_src", dataset="mv_train", view="sample1000"),
    ],
    extractors=[
        BoVWExtractorConfig(name="bovw_ext", vocab_size=256, batch_size=32),
    ],
    workflows=[cleaning],
    tasks=[task],
)

## Run the matrix

`run_task` checks every run's configuration first, then runs the entry once per combination and returns one
`MatrixResult`. The runs share one draw of the source's view, and the cache computes the image statistics, the BoVW
embeddings and the clusters once for all nine. See {doc}`../how_to/reuse_results_with_cache` to persist them across
runs. To run a matrix from a YAML configuration in a container, see {doc}`../how_to/containerized_workflows`.

In [ ]:
result = run_task(config, task, cache_dir=Path("./cache"))
print(type(result).__name__, len(result.runs), "runs")

## Read the comparison table

`report(detailed=False)` is the short form the console prints: the matrix's health, then a table with a row per run.
Each row gives the run's number, the values it ran with, its health (`[!!]` where it warned), and a column per
finding, each cell the finding's severity marker and brief. The table is wide, so `width=160` gives it room. At the
console's 80 columns it wraps its cells further.

In [ ]:
print(result.report(detailed=False, width=160))

`result.to_html(detailed=False)` renders the same comparison as one HTML page. `result.to_html()` adds each run's full
report below it. The page follows your system's dark mode setting, so here it may not match this site's theme. See
{doc}`view_html_reports` for the rest of the page.

In [ ]:
# The page styles its whole document, so the docs show it in an iframe of its own.
import html

from IPython.display import display


def show(page: str, height: int = 560) -> None:
    """*page* in an iframe, so its stylesheet and script stay inside it."""
    frame = (
        f'<iframe srcdoc="{html.escape(page, quote=True)}" title="HTML report" loading="lazy" '
        f'style="width:100%;height:{height}px;border:1px solid #d1d9e0;border-radius:6px"></iframe>'
    )
    display({"text/html": frame}, raw=True)

In [ ]:
show(result.to_html(detailed=False))

### What the table shows

Every run warns, with 3 warnings each and 27 in all. Image Outliers, Class Outliers and Image Duplicates are past
their thresholds in every run, and Class Imbalance is `info`.

The `outliers.outlier_threshold` setting changes the outlier findings. The adaptive method flags 157 images (15.7%) at
2.5, 112 (11.2%) at 3.5 and 96 (9.6%) at 4.5. The count falls by 45 images from 2.5 to 3.5, then by 16 from 3.5 to
4.5, so it changes less the higher the threshold. The class with the largest share of outliers changes with it: 2S19
MSTA (25.0%) at 2.5, and Tornado at 3.5 (20.4%) and 4.5 (16.3%). Even at 4.5, 9.6% of the sample is flagged, over the
3% at which `image-outliers` warns by default, and 23 of the 24 classes are over the classwise threshold at every
setting.

The `duplicates.cluster_sensitivity` setting changes nothing on this sample. Runs 1, 2 and 3 differ only in it, and
their rows are identical. So are runs 4 to 6, and runs 7 to 9. Every run counts 4 images as exact duplicates (0.4%)
and 6 as near duplicates (0.6%). The "Read one run in full" section shows why.

The label distribution depends on neither setting. It is 24 classes, 1,000 items and an imbalance of 3.0:1 in every
run.

## Read one run in full

Each run's result is the result the task would return run alone with those values. `result.runs` holds the runs in
order, each with its number, its values and its result, here a `ChainResult`. Run 4 is the fourth:

In [ ]:
run = result.runs[3]
print(run.number, run.label)
for finding in run.result.findings:
    print(f"{finding.severity:<8} {finding.title:<20} {finding.brief}")

Its findings are row 4 of the table. Its steps hold each step's output, as a lone `quality` run's do. The `duplicates`
step's output is DataEval's duplicates output, and its `data()` lists the groups behind the Image Duplicates finding,
with the methods that found each:

In [ ]:
print(run.result.steps["duplicates"].output.data())

The two exact groups are pairs that `xxhash` matched. Of the three near groups, one is a pair the perceptual hashes
(`dhash`, `phash`) found, and the other two are the same two exact pairs, found again by the cluster pass. The cluster
pass is the only part of duplicate detection that `duplicates.cluster_sensitivity` changes, and at 0.5, 2.0 and 3.0
alike it finds the exact copies and nothing else, so the count does not move.

`result.report(detailed=True)`, the default, adds each run's full report under the table, in a `Runs` section, each
headed with its number and values, as `dataeval-flow -v` prints it.

## Choose the settings

From this table:

- `outliers.outlier_threshold` matters. The count is still falling at 4.5, but by about a third of what it fell from
  2.5 to 3.5, so a value from 3.5 to 4.5 depends less on the exact choice than one below 3.5. Even 4.5 flags more than
  the default 3% warning, so look at the flagged images, as {doc}`data_cleaning` does, before raising
  `checks.image-outliers.warning` for a collection this varied. See {doc}`../how_to/configure_outlier_detection` for
  the outlier settings and how to choose them.
- `duplicates.cluster_sensitivity` does not matter here, because 0.5, 2.0 and 3.0 give the same groups. Keep any of
  them, or leave it and `duplicates.cluster_algorithm` unset to skip the cluster pass, which found no group the hashes
  missed.

## Conclusion

In this tutorial, you learned how to:

- Vary a `quality` entry's settings with a `matrix` on its task.
- Run every combination over one draw of the data, sharing the statistics, embeddings and clusters.
- Read the comparison table, and reach each run's result.
- Choose quality settings from the settings that change the findings.

## Next steps

- {doc}`data_cleaning` — Apply the chosen settings in an operational cleaning run.
- {doc}`../how_to/run_a_matrix` — Vary any workflow's or evaluator's settings, its sources, or its extractor, and
  learn every way to write a matrix's values and keys.
- {doc}`../concepts/Reproducibility` — Learn how config-keyed caching reuses embeddings and statistics across runs.

## On your own

Repeat the analysis on your own data by changing these settings:

- Set `path` on `HuggingFaceDatasetConfig` to your own image classification dataset in class-per-directory format, and
  keep `task="image_classification"`.
- Change the `size` param of the `Limit` operation in the view, or remove the operation to use every image. Keep the
  `Shuffle` operation before it.
- Change `seed` on the pipeline, which each run starts from, and the `seed` param of the `Shuffle` operation to draw a
  different sample.
- Change the values under `matrix`. List a `[method, bound]` pair for each value of `outliers.outlier_threshold`, and
  list the values to try for `duplicates.cluster_sensitivity`.
- Add a key to `matrix`, such as `checks.image-outliers.warning`. The grid crosses every key with every other, so each
  key you add multiplies the number of runs.
- Change `flags` in the `outliers` settings of the entry to use fewer of the `dimension`, `pixel` and `visual`
  statistics.
- Change `vocab_size` on `BoVWExtractorConfig`, which is 256 here, or replace the extractor config.